# Calling aviso-core through the Aviso SDK

How to reach `basic_results`, `deals_results` and `drilldown_fields` on aviso-core
the same way the legacy GBM shell is used: log in, then call. No `Internal-Api-Key`.

Runs on the Jupyter kernel as-is. Replace the placeholders in angle brackets.

## 1. Log in as usual

Nothing new here — this is the same shell everyone already uses.

In [ ]:
import json
import os

from gshell import GnanaShell

src = GnanaShell('https://app.aviso.com')
src.signin(username='<you>@administrative.domain')
src.switch_to('<tenant>.com')

## 2. One-time: register your public key on aviso-core

`signin()` authenticates by signing a timestamp with your private key, so the
service has to know the matching public one. Every Aviso service needs this
once per user — see `python-sdk/src/signin setup for all services.ipynb`.

Log in with your password first, then upload the key. Skip this cell once it
has been done.

In [ ]:
from avisosdk import connect_sdk

CORE = 'https://aviso-core-dev.aviso.com'

setup = connect_sdk(CORE)
setup.login(username='<you>@administrative.domain')
print(setup.api('/account/keys?action=set', os.path.expanduser('~/.ssh/id_rsa.pub')))

## 3. Get a shell on aviso-core

Once the tenant's `microservices_info.gbm_service.host` points at aviso-core,
`src.gbm` is already this shell and you can skip the cell below — that is the
intended path and it needs no code change on the SDK side.

Connect directly while testing, before the config is switched over.

In [ ]:
core = connect_sdk(CORE)
core.signin(username='<you>@administrative.domain', tenant_name='<tenant>.com')

print(core.me()['username'])

## 4. Call the three APIs

In [ ]:
period = src.current_period()['mnemonic']
deal_id = '<opportunity id>'

basic = core.api('/gbm/basic_results?period=%s&id_list=%s' % (period, deal_id), None)
print(len(basic))

In [ ]:
deals = core.api('/gbm/deals_results?period=%s&fields=as_of_Amount_USD' % period, None)
print(sorted(deals.keys()))

In [ ]:
drilldown = core.api('/gbm/v2/drilldown_fields?period=%s&owner_mode=false' % period,
                     json.dumps({'fields_list': ['Owner']}))
print(drilldown)

## 5. The static key no longer opens anything

Any value gives the same answer: without a session or a per-tenant
`Access-Token` the service does not serve the request.

In [ ]:
import requests

refused = requests.get(CORE + '/gbm/basic_results?period=' + period,
                       headers={'Internal-Api-Key': '<any value>',
                                'X-Tenant-Name': '<tenant>.com'})
print(refused.status_code, refused.text)  # 401

## Notes

- The tenant comes from your session, not from a header. `switch_to()` moves it;
  `X-Tenant-Name` is ignored on the session path.
- The session is signed into a cookie, so it is not tied to the task that served
  the login and it cannot be revoked server side — log out when you are done.
- `core` exposes `.api()` plus the three endpoints above. `results()`,
  `combined_results()`, `target_spec()` and friends live in the legacy GBM shell
  and appear here as those APIs are migrated.
- Jobs that cannot hold a session use an `Access-Token` bound to one tenant
  instead, not a shared secret.